# Debugging the stroke restorer: what does each layer actually learn?

The restorer used in runs 3 and 4, M1 and M2 (`StrokeRestorer` in `scripts/train_accepted_pairs.py`) is small: three convolutions.

```
input (grey) -> conv1: 64 filters 9x9 -> ReLU -> conv2: 32 filters 5x5 -> ReLU -> conv3: 5x5 -> correction
output = input + correction            (the network only predicts what to ADD to each pixel)
```

This notebook runs a model **one layer at a time** on real far-away test lines and shows:

| Part | Question it answers |
|---|---|
| A | What did the whole network change, and was it the right change? |
| B | Did it open the gaps between strokes, or only darken them? |
| C | What patterns does layer 1 look for? (its 64 filters) |
| D | Which of those patterns fire on our line? Which are dead (never fire)? |
| E | What do the 32 layer-2 features respond to? |
| F | Which layer-2 features build the correction? |
| G | Where does the correction go: strokes, next to strokes, or empty board? (all test pairs, every model) |
| H | The same line through every model |

Nothing here trains or changes a model; it only reads the saved checkpoints.

**Two terms used below**
- **Activation / feature map**: the output image of one filter. Bright = "my pattern is here".
- **Receptive field**: how many input pixels one output pixel can "see". 9 + 4 + 4 -> 17 x 17 px for this network. A letter bigger than that cannot be recognised as a whole by the network, only its local edges.

In [12]:
# ============================ settings ============================
MODELS = {   # name -> checkpoint (all StrokeRestorer, trained on the same normalised grey input)
    "run3 (ink-matched)": "outputs/_sr_review/run3_ink_matched/accepted_pair_restorer.pt",
    "run4 (pixel loss)": "outputs/_sr_review/run4_pixel_loss/accepted_pair_restorer.pt",
    "M1 (stroke loss)": "models/superres/m1_stroke_loss.pt",
    "M2 (recognition loss)": "models/superres/m2_recognition_loss.pt",
}
MAIN_MODEL = "M1 (stroke loss)"      # the model looked at in detail (parts A-F)
MANIFEST = "outputs/paired_dataset_airdrop/airdrop_test_manifest.csv"   # real far -> close pairs
PAIR_GROUP = "airdrop_B3"            # the CNN board: never used in training (unseen-board test)
SHOW_PAIRS = ["IMG_4530__IMG_4543__B3_L01_8c6e5008",    # "CNN Architecture"
              "IMG_4515__IMG_4543__B3_L04_86eb6baa",    # "Visualising CNN"
              "IMG_4526__IMG_4543__B3_L06_a9931c56"]    # "Visualise feature layer"
DEVICE = "cpu"                       # tiny model: CPU is fine
SAVE_FIGURES = True                  # also save every figure to outputs/_sr_review/activations/

In [13]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "scripts").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the whiteboard-ocr repo folder")


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "scripts"))
import pair_data                      # noqa: E402  (same normalisation and masks as training)
import train_accepted_pairs as tap    # noqa: E402  (same model class and loading as training)

FIG_DIR = REPO / "outputs" / "_sr_review" / "activations"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    if SAVE_FIGURES:
        fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")


models = {}
for name, path in MODELS.items():
    p = REPO / path
    if not p.exists():
        print(f"skip {name}: {path} not found")
        continue
    try:
        models[name] = tap.load_model(p, DEVICE)          # refuses checkpoints made with another normalisation
    except ValueError as e:
        print(f"skip {name}: {e}")
        continue
    n = sum(v.numel() for v in models[name].parameters())
    print(f"loaded {name:<24} {n:,} weights  <- {path}")
assert MAIN_MODEL in models, f"MAIN_MODEL {MAIN_MODEL!r} could not be loaded"

rows = [r for r in csv.DictReader(open(REPO / MANIFEST, encoding="utf-8")) if r["group_id"] == PAIR_GROUP]
by_id = {r["pair_id"]: r for r in rows}
missing = [p for p in SHOW_PAIRS if p not in by_id]
assert not missing, f"not in {MANIFEST} ({PAIR_GROUP}): {missing}"
print(f"{len(rows)} test pairs on {PAIR_GROUP}; showing {len(SHOW_PAIRS)} in detail")

loaded run3 (ink-matched)       69,251 weights  <- outputs/_sr_review/run3_ink_matched/accepted_pair_restorer.pt
loaded run4 (pixel loss)        69,251 weights  <- outputs/_sr_review/run4_pixel_loss/accepted_pair_restorer.pt
loaded M1 (stroke loss)         69,251 weights  <- models/superres/m1_stroke_loss.pt
loaded M2 (recognition loss)    69,251 weights  <- models/superres/m2_recognition_loss.pt
24 test pairs on airdrop_B3; showing 3 in detail


## Running the network one layer at a time

`trace` does exactly what `model(x)` does, but keeps every intermediate image. The `assert` checks that the result is identical to the normal forward pass, so what you see is what the model really computes.

The model gets the same input as in training and testing: the far photo's line, enlarged to the clear photo's size, board flattened to white (`normalise_board`), grey in all 3 channels.

In [14]:
def trace(model, img):
    """img: normalised grey canvas (H, W, 3), float in [0, 1]. Returns every intermediate image."""
    b = model.body
    x = tap.to_tensor(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        z1 = b[0](x)                     # conv1: 64 filters, before ReLU
        a1 = torch.relu(z1)              # what layer 1 passes on (negative = "pattern absent" -> 0)
        z2 = b[2](a1)                    # conv2: 32 filters
        a2 = torch.relu(z2)
        corr = b[4](a2)                  # conv3: the correction to ADD to the input (3 channels)
        out = torch.clamp(x + corr, 0.0, 1.0)
        assert float((out - model(x)).abs().max()) < 1e-5, "trace differs from model(x)"
    np_ = lambda t: t[0].cpu().numpy()
    return dict(conv1=np_(z1), relu1=np_(a1), conv2=np_(z2), relu2=np_(a2),
                correction=np_(corr).mean(0),                 # what the network ASKS for (before clipping)
                change=(np_(out) - np_(x)).mean(0),           # what really changes: output - input (after clipping to 0..1)
                output=np_(out).mean(0))


def receptive_field(model):
    ks = [m.weight.shape[-1] for m in model.body if hasattr(m, "weight")]
    return 1 + sum(k - 1 for k in ks)


pairs = {}
for pid in SHOW_PAIRS:
    inp, tgt, ok = tap.load_pair(by_id[pid])
    pairs[pid] = dict(inp=inp, tgt=tgt, ok=ok, text=by_id[pid].get("transcription_verified") or by_id[pid]["transcription_draft"])

RF = receptive_field(models[MAIN_MODEL])
for pid, p in pairs.items():
    ink_rows = np.nonzero(pair_data.ink_mask(p["tgt"], grow=0).any(axis=1))[0]
    ink_h = int(ink_rows[-1] - ink_rows[0] + 1) if len(ink_rows) else 0
    print(f"{p['text']!r:<28} canvas {p['inp'].shape[1]} x {p['inp'].shape[0]} px | writing about {ink_h} px tall "
          f"| the network sees {RF} x {RF} px around each pixel")

'CNN Architecture'           canvas 541 x 105 px | writing about 66 px tall | the network sees 17 x 17 px around each pixel
'Visualising CNN'            canvas 380 x 106 px | writing about 76 px tall | the network sees 17 x 17 px around each pixel
'Visualise feature layer'    canvas 425 x 81 px | writing about 60 px tall | the network sees 17 x 17 px around each pixel


## A. What the network changed, and whether it was the right change

Rows, top to bottom:
1. **input** (far photo, enlarged); the square shows the receptive field (all the network can see for one pixel);
2. **target** (clear photo);
3. **output** of the network;
4. **change** = output - input (after the output is clipped to 0..1, so only changes that really happen). **Blue = made darker (added ink), red = made lighter (removed ink)**;
5. **error before** = |input - target|; 6. **error after** = |output - target|. Rows 5 and 6 share one colour scale, so darker in row 6 than in row 5 means the network fixed something there.

The green outline is the target's ink. A useful correction is blue *inside* the green outline where the input is too light, and red in the gaps where the input is too dark.

In [15]:
def show_pair(model_name, pid, ax_col=None):
    p, t = pairs[pid], trace(models[model_name], pairs[pid]["inp"])
    inp, tgt = p["inp"][..., 0], p["tgt"][..., 0]
    ink = pair_data.ink_mask(p["tgt"], grow=0)
    err0, err1 = np.abs(inp - tgt), np.abs(t["output"] - tgt)
    lim = max(0.02, float(np.percentile(np.abs(t["change"]), 99.5)))
    emax = float(np.percentile(err0, 99.5))
    panels = [("input (far photo, enlarged)", inp, "gray", 0, 1), ("target (clear photo)", tgt, "gray", 0, 1),
              ("output", t["output"], "gray", 0, 1),
              (f"change = output - input (blue = made darker, red = made lighter), +/-{lim:.2f}", t["change"], "bwr", -lim, lim),
              ("error before |input - target|", err0, "magma", 0, emax),
              ("error after |output - target|", err1, "magma", 0, emax)]
    fig, axes = plt.subplots(len(panels), 1, figsize=(12, len(panels) * (12 * inp.shape[0] / inp.shape[1] + 0.4) + 0.4))
    for a, (title, im, cmap, lo, hi) in zip(axes, panels):
        a.imshow(im, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
        if title.startswith(("change", "error")):
            a.contour(ink, levels=[0.5], colors="lime", linewidths=0.6)
        a.set_title(title, fontsize=9, loc="left"); a.set_xticks([]); a.set_yticks([])
    y0 = max(0, inp.shape[0] // 2 - RF // 2)
    axes[0].add_patch(plt.Rectangle((8, y0), RF, RF, fill=False, ec="red", lw=1.2))
    ok = p["ok"]
    m_ink = pair_data.ink_mask(p["tgt"], grow=3) & ok
    fig.suptitle(f"{model_name} | truth: {p['text']!r} | PSNR on strokes: input {pair_data.psnr(p['inp'], p['tgt'], m_ink):.2f} dB "
                 f"-> output {pair_data.psnr(np.repeat(t['output'][..., None], 3, 2), p['tgt'], m_ink):.2f} dB", fontsize=10)
    plt.tight_layout()
    return fig, t


for pid in SHOW_PAIRS:
    fig, _ = show_pair(MAIN_MODEL, pid)
    save(fig, f"A_{MAIN_MODEL.split()[0]}_{pid[:30]}"); plt.show()

<Figure size 1200x1677.41 with 6 Axes>

<Figure size 1200x2288.42 with 6 Axes>

<Figure size 1200x1652.24 with 6 Axes>

## B. Did it open the gaps, or only darken the strokes?

One row of pixels through the writing: the row that crosses the most separate strokes in the target (not an underline). Low = ink, high = board.
- A **sharper** result has deeper valleys (strokes) **and** higher peaks between them (gaps) than the input, i.e. it moves toward the target in both directions.
- If the output's valleys get deeper but the peaks between close strokes stay low, the network is **thickening blobs**, not separating strokes.

In [16]:
fig, axes = plt.subplots(len(SHOW_PAIRS), 1, figsize=(13, 2.6 * len(SHOW_PAIRS)))
for a, pid in zip(np.atleast_1d(axes), SHOW_PAIRS):
    p, t = pairs[pid], trace(models[MAIN_MODEL], pairs[pid]["inp"])
    ink_t = p["tgt"][..., 0] < 0.75
    row = int(np.argmax((np.diff(ink_t.astype(int), axis=1) == 1).sum(axis=1)))   # row crossing the most strokes
    x = np.arange(p["inp"].shape[1])
    a.plot(x, p["tgt"][row, :, 0], color="black", lw=1.3, label="target (clear)")
    a.plot(x, p["inp"][row, :, 0], color="tab:orange", lw=1.0, label="input (far)")
    a.plot(x, t["output"][row], color="tab:blue", lw=1.0, label=f"output ({MAIN_MODEL})")
    a.fill_between(x, 0, 1, where=p["tgt"][row, :, 0] < 0.75, color="lime", alpha=0.12, label="target ink (< 0.75)")
    a.set_ylim(0, 1.05); a.set_title(f"{p['text']!r}, pixel row {row}", fontsize=9, loc="left")
    a.set_ylabel("brightness")
    a.legend(fontsize=7, ncol=4, loc="lower right")
axes = np.atleast_1d(axes); axes[-1].set_xlabel("pixel column")
plt.tight_layout(); save(fig, f"B_profiles_{MAIN_MODEL.split()[0]}"); plt.show()

<Figure size 1300x780 with 3 Axes>

## C. Layer 1: what patterns does the network look for?

Each small square is one of the 64 first-layer filters (9 x 9 px). The input is the same grey image in all 3 channels, so the three colour slices of a filter act together: we show their **sum**, which is exactly what the filter applies to our grey input.

How to read them: **blue** where the filter wants dark (ink), **red** where it wants bright (board), white = ignores.
- A blue bar next to a red bar = an **edge detector** for strokes in that direction.
- A blue blob with a red ring = a **dot / thin stroke** detector.
- Noise with no structure = a filter that learned little.

Filters are sorted by how strongly they fire on all test pairs (first = most used). Grey titles = **dead**: they never fire on any test pixel, so they contribute nothing.

In [17]:
def channel_usage(model, layer):
    """Mean activation of every channel over all test pairs, and whether it ever fires (> 0)."""
    sums, peaks, n = 0.0, None, 0
    for r in rows:
        inp, _, ok = tap.load_pair(r)
        a = trace(model, inp)[layer]                         # (C, H, W)
        m = a[:, ok]
        sums = sums + m.mean(axis=1); n += 1
        peaks = m.max(axis=1) if peaks is None else np.maximum(peaks, m.max(axis=1))
    return sums / n, peaks > 1e-6


usage1, alive1 = channel_usage(models[MAIN_MODEL], "relu1")
usage2, alive2 = channel_usage(models[MAIN_MODEL], "relu2")
print(f"{MAIN_MODEL}: layer 1 dead filters {int((~alive1).sum())} / {len(alive1)}, "
      f"layer 2 dead filters {int((~alive2).sum())} / {len(alive2)} (on {len(rows)} test pairs)")

W1 = models[MAIN_MODEL].body[0].weight.detach().cpu().numpy().sum(axis=1)       # (64, 9, 9)
order1 = np.argsort(-usage1)
fig, axes = plt.subplots(8, 8, figsize=(9, 9.6))
for a, c in zip(axes.ravel(), order1):
    lim = float(np.abs(W1[c]).max()) or 1.0
    a.imshow(W1[c], cmap="bwr", vmin=-lim, vmax=lim)
    a.set_title(f"#{c} {usage1[c]:.2f}", fontsize=7, color="black" if alive1[c] else "grey")
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"{MAIN_MODEL}: the 64 layer-1 filters (sum over the 3 input channels), sorted by use", fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.97]); save(fig, f"C_filters1_{MAIN_MODEL.split()[0]}"); plt.show()

M1 (stroke loss): layer 1 dead filters 0 / 64, layer 2 dead filters 0 / 32 (on 24 test pairs)


<Figure size 900x960 with 64 Axes>

## D. Layer 1 on our line: which patterns fire where?

The 64 feature maps of layer 1 (after ReLU) for the first line, in the same order as the filters above. Bright = the filter's pattern is present there.
Typical findings: some maps light up on **all** ink (a "where is ink" map), some only on one side of strokes (edge maps), some on the board texture (noise / glare maps: anything the network does with those changes the board, not the letters).

In [18]:
pid0 = SHOW_PAIRS[0]
t0 = trace(models[MAIN_MODEL], pairs[pid0]["inp"])


def show_maps(maps, order, alive, usage, title, ncols=4, name=""):
    n = len(order)
    nrows = int(np.ceil(n / ncols))
    h, w = maps.shape[1:]
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, (4.2 * h / w + 0.35) * nrows))
    for a, c in zip(axes.ravel(), order):
        a.imshow(maps[c], cmap="viridis", vmin=0, vmax=max(float(maps[c].max()), 1e-6))
        a.set_title(f"#{c} mean {usage[c]:.2f}" + ("" if alive[c] else " (dead)"), fontsize=7,
                    color="black" if alive[c] else "grey")
        a.set_xticks([]); a.set_yticks([])
    for a in axes.ravel()[n:]:
        a.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, 0.985]); save(fig, name); plt.show()


show_maps(t0["relu1"], order1, alive1, usage1,
          f"{MAIN_MODEL}: layer-1 feature maps on {pairs[pid0]['text']!r} (each scaled to its own max)",
          name=f"D_maps1_{MAIN_MODEL.split()[0]}")

<Figure size 1680x1864.25 with 64 Axes>

## E. Layer 2: combinations of layer-1 patterns

Each of the 32 layer-2 filters combines all 64 layer-1 maps over a 5 x 5 area. They usually respond to bigger structures: whole stroke widths, stroke ends, places where two strokes come close.

In [19]:
order2 = np.argsort(-usage2)
show_maps(t0["relu2"], order2, alive2, usage2,
          f"{MAIN_MODEL}: layer-2 feature maps on {pairs[pid0]['text']!r} (each scaled to its own max)",
          name=f"E_maps2_{MAIN_MODEL.split()[0]}")

<Figure size 1680x932.126 with 32 Axes>

## F. Which layer-2 features build the correction?

The last layer adds up the 32 layer-2 maps, each through its own 5 x 5 filter, plus a constant (bias). Because it is a sum, we can split the correction into **one part per feature**, and see which features do the work.
- Bar chart: how much each feature contributes (mean |part|).
- Maps: the 8 largest parts (blue = darker, red = lighter). These add up to the network's raw correction *before* clipping: on the white board, a "lighter" part may be cut off at 1.0 and change nothing.
If one or two features produce almost everything and they are "darken wherever there is ink" maps, the network is doing one simple thing: making ink darker.

In [20]:
conv3 = models[MAIN_MODEL].body[4]
a2 = torch.from_numpy(np.ascontiguousarray(t0["relu2"]))[None]                    # (1, 32, H, W)
W3 = conv3.weight.detach().cpu()                                                    # (3, 32, 5, 5)
parts = np.stack([F.conv2d(a2[:, c:c + 1], W3[:, c:c + 1], padding=W3.shape[-1] // 2)[0].numpy().mean(0)
                  for c in range(W3.shape[1])])                                     # (32, H, W)
bias = float(conv3.bias.detach().cpu().numpy().mean())
total = parts.sum(0) + bias
assert np.abs(total - t0["correction"]).max() < 1e-4, "parts do not add up to the correction"
energy = np.abs(parts).mean(axis=(1, 2))
top = np.argsort(-energy)[:8]
print(f"bias (constant added everywhere): {bias:+.4f} | top 3 features give "
      f"{energy[top[:3]].sum() / energy.sum():.0%} of the total contribution")

fig = plt.figure(figsize=(14, 2.6))
plt.bar(range(len(energy)), energy, color=["tab:blue" if alive2[c] else "lightgrey" for c in range(len(energy))])
plt.xlabel("layer-2 feature #"); plt.ylabel("mean |contribution|"); plt.title("contribution of each layer-2 feature to the correction", fontsize=10)
save(fig, f"F_contrib_bar_{MAIN_MODEL.split()[0]}"); plt.show()

lim = max(1e-3, float(np.percentile(np.abs(parts[top]), 99.5)))
ink0 = pair_data.ink_mask(pairs[pid0]["tgt"], grow=0)
fig, axes = plt.subplots(4, 2, figsize=(15, 4 * (7.5 * parts.shape[1] / parts.shape[2] + 0.4) + 0.4))
for a, c in zip(axes.ravel(), top):
    a.imshow(parts[c], cmap="bwr", vmin=-lim, vmax=lim)
    a.contour(ink0, levels=[0.5], colors="lime", linewidths=0.5)
    a.set_title(f"feature #{c}: {energy[c] / energy.sum():.0%} of the contribution", fontsize=8)
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"{MAIN_MODEL}: the 8 biggest parts of the correction on {pairs[pid0]['text']!r} (green = target ink)", fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.97]); save(fig, f"F_parts_{MAIN_MODEL.split()[0]}"); plt.show()

bias (constant added everywhere): +0.0000 | top 3 features give 42% of the total contribution


<Figure size 1400x260 with 1 Axes>

<Figure size 1500x782.255 with 8 Axes>

## G. Where does the correction go? (all test pairs, every model)

Every pixel of every test pair on this board is put into one of three regions, from the **target's** ink:

| Region | Meaning |
|---|---|
| **stroke** | target ink |
| **near stroke** | within 3 px of target ink, not ink: stroke edges and the gaps between close strokes |
| **far board** | everything else |

For each region, the table shows:
- `share of pixels`;
- `share of correction`: how much of the real change (sum of |output - input|) lands there;
- `mean correction`: mean of output - input; negative = made darker, positive = made lighter;
- `error before` / `error after`: mean |input - target| and mean |output - target|.

**How to read it:** strokes are about 10% of a line. If most of the correction lands on the far board, the network is mostly cleaning the board. If `error after` is above `error before` in a region, it made that region worse.

In [21]:
def region_stats(model):
    acc = {}
    for r in rows:
        inp, tgt, ok = tap.load_pair(r)
        t = trace(model, inp)
        ink0 = pair_data.ink_mask(tgt, grow=0)
        near = pair_data.ink_mask(tgt, grow=3) & ~ink0
        regions = {"stroke": ink0 & ok, "near stroke": near & ok, "far board": ~(ink0 | near) & ok}
        g_in, g_tg = inp[..., 0], tgt[..., 0]
        for k, m in regions.items():
            a = acc.setdefault(k, dict(pixels=0, abs_corr=0.0, corr=0.0, err0=0.0, err1=0.0))
            a["pixels"] += int(m.sum()); a["abs_corr"] += float(np.abs(t["change"][m]).sum())
            a["corr"] += float(t["change"][m].sum())
            a["err0"] += float(np.abs(g_in - g_tg)[m].sum()); a["err1"] += float(np.abs(t["output"] - g_tg)[m].sum())
    tot_px, tot_c = sum(a["pixels"] for a in acc.values()), sum(a["abs_corr"] for a in acc.values())
    return pd.DataFrame({k: {"share of pixels": a["pixels"] / tot_px, "share of correction": a["abs_corr"] / max(tot_c, 1e-12),
                             "mean correction": a["corr"] / max(a["pixels"], 1), "error before": a["err0"] / max(a["pixels"], 1),
                             "error after": a["err1"] / max(a["pixels"], 1)} for k, a in acc.items()}).T


tables = []
for name, m in models.items():
    s = region_stats(m)
    s.insert(0, "model", name)
    tables.append(s.reset_index().rename(columns={"index": "region"}))
regions = pd.concat(tables, ignore_index=True)
regions.to_csv(FIG_DIR / "G_where_the_correction_goes.csv", index=False)
shown = regions.copy()
for col in ("share of pixels", "share of correction"):
    shown[col] = shown[col].map(lambda v: f"{v:.0%}")
for col in ("mean correction", "error before", "error after"):
    shown[col] = shown[col].map(lambda v: f"{v:+.4f}" if col == "mean correction" else f"{v:.4f}")
display(shown)

dead = pd.DataFrame([dict(model=n, **{f"layer {k} dead": f"{int((~alive).sum())} / {len(alive)}"
                                       for k, alive in ((1, channel_usage(m, 'relu1')[1]), (2, channel_usage(m, 'relu2')[1]))})
                     for n, m in models.items()])
display(dead)

,region,model,share of pixels,share of correction,mean correction,error before,error after
0,stroke,run3 (ink-matched),9%,19%,+0.0238,0.1353,0.1393
1,near stroke,run3 (ink-matched),17%,23%,+0.0154,0.1298,0.1160
2,far board,run3 (ink-matched),73%,58%,+0.0090,0.0144,0.0103
3,stroke,run4 (pixel loss),9%,23%,+0.0694,0.1353,0.1560
4,near stroke,run4 (pixel loss),17%,40%,+0.0635,0.1298,0.0748
5,far board,run4 (pixel loss),73%,36%,+0.0136,0.0144,0.0093
6,stroke,M1 (stroke loss),9%,12%,+0.0267,0.1353,0.1385
7,near stroke,M1 (stroke loss),17%,33%,+0.0460,0.1298,0.0906
8,far board,M1 (stroke loss),73%,55%,+0.0182,0.0144,0.0097
9,stroke,M2 (recognition loss),9%,29%,+0.0500,0.1353,0.1501


,model,layer 1 dead,layer 2 dead
0,run3 (ink-matched),0 / 64,0 / 32
1,run4 (pixel loss),0 / 64,0 / 32
2,M1 (stroke loss),0 / 64,0 / 32
3,M2 (recognition loss),0 / 64,3 / 32


## H. The same line through every model

Change maps (output - input) side by side (same colour scale for all models): do the models differ in *where* they act, or only in *how much*?

In [22]:
traces = {n: trace(m, pairs[pid0]["inp"]) for n, m in models.items()}
lim = max(0.02, max(float(np.percentile(np.abs(t["change"]), 99.5)) for t in traces.values()))
inp0, tgt0 = pairs[pid0]["inp"][..., 0], pairs[pid0]["tgt"][..., 0]
n = 2 + 2 * len(traces)
h, w = inp0.shape
fig, axes = plt.subplots(n, 1, figsize=(12, n * (12 * h / w + 0.35)))
axes[0].imshow(inp0, cmap="gray", vmin=0, vmax=1); axes[0].set_title("input", fontsize=8, loc="left")
axes[1].imshow(tgt0, cmap="gray", vmin=0, vmax=1); axes[1].set_title("target", fontsize=8, loc="left")
for i, (name, t) in enumerate(traces.items()):
    axes[2 + 2 * i].imshow(t["output"], cmap="gray", vmin=0, vmax=1)
    axes[2 + 2 * i].set_title(f"{name}: output", fontsize=8, loc="left")
    axes[3 + 2 * i].imshow(t["change"], cmap="bwr", vmin=-lim, vmax=lim)
    axes[3 + 2 * i].contour(pair_data.ink_mask(pairs[pid0]["tgt"], grow=0), levels=[0.5], colors="lime", linewidths=0.5)
    axes[3 + 2 * i].set_title(f"{name}: change = output - input (blue = darker, red = lighter, +/-{lim:.2f})", fontsize=8, loc="left")
for a in axes:
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"truth: {pairs[pid0]['text']!r}", fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.985]); save(fig, "H_models_side_by_side"); plt.show()

<Figure size 1200x2679.02 with 10 Axes>

## What to look for (debug checklist)

1. **Receptive field vs letter size** (printed above part A). If the writing is several times taller than the 17 px the network can see, it can only sharpen *edges*. It cannot know *which letter* it is fixing, so it cannot restore a missing "u"/"a" gap. That is a limit of the architecture, not of the loss.
2. **Part A, row 4**: is the correction mostly blue on the green outline (darkening strokes), red in the gaps (opening them), or spread over the board?
3. **Part B**: do the gaps between close strokes rise toward the target (separation) or stay low (blob)?
4. **Part C / D**: dead filters = wasted capacity. Many dead filters means the model uses only a few patterns.
5. **Part F**: if one or two features build most of the correction, the model has learned one simple rule (often "darker where there is ink").
6. **Part G**: compare `share of correction` with `share of pixels`, and `error after` with `error before`, per region and per model. A model can lower the error on strokes while raising it near strokes (thicker strokes that fill the gaps): that pattern matches "higher PSNR, same or worse reading".

Write down what you see per model; the next change (bigger receptive field, recognition loss, ...) should target the specific failure found here.